# Fermi-Hubbard Model Simulation via VQE

## Overview

This notebook simulates the **3-site Fermi-Hubbard model** using the **Variational Quantum Eigensolver (VQE)** algorithm implemented with Qiskit.

The Fermi-Hubbard model captures two competing effects in a lattice of fermions:
- **Hopping (kinetic energy):** Fermions tunnel between adjacent lattice sites, governed by the hopping parameter $J$.
- **On-site interaction (potential energy):** Two fermions occupying the same site interact with energy $U$.

The full 6-qubit Hamiltonian (Jordan-Wigner encoding, 3 sites × 2 spins) is:

$$H = -\frac{J}{2}\left[(X_1X_3 + Y_1Y_3)Z_2 + (X_3X_5 + Y_3Y_5)Z_4 + (X_2X_4 + Y_2Y_4)Z_3 + (X_4X_6 + Y_4Y_6)Z_5\right]$$
$$+ \frac{U}{4}\left[(I - Z_1 - Z_2 + Z_1Z_2) + (I - Z_3 - Z_4 + Z_3Z_4) + (I - Z_5 - Z_6 + Z_5Z_6)\right]$$

We will:
1. Construct the Hamiltonian as a `SparsePauliOp`
2. Compute the exact ground state energy via classical diagonalization (for $U=0$)
3. Run VQE with an `EfficientSU2` ansatz and compare to the exact result
4. Study convergence as $J$ varies over $[1.0, 5.0]$
5. *(Optional)* Study convergence as $U$ varies over $\{0, 0.5, 1.0\}$

In [ ]:
# Imports and setup
import numpy as np
import matplotlib.pyplot as plt
from fermi_hubbard_helpers import (
    build_hamiltonian,
    exact_ground_energy,
    run_vqe,
    make_ansatz,
    sweep_J,
    sweep_U,
    compute_error_flag,
    SweepResult,
)
from qiskit_algorithms.optimizers import SLSQP

---
## Section 1: Hamiltonian Construction

### Physical Background

The Fermi-Hubbard Hamiltonian describes fermions on a lattice with two competing terms:

- **Hopping term** (scaled by $-J/2$): Represents kinetic energy. Fermions can tunnel between nearest-neighbor sites. In the Jordan-Wigner encoding, hopping between sites $i$ and $j$ maps to correlated Pauli strings of the form $X_iX_jZ_{\text{between}}$ and $Y_iY_jZ_{\text{between}}$, where the $Z$ operators on intermediate qubits enforce fermionic anti-commutation.

- **On-site interaction term** (scaled by $U/4$): Represents potential energy. When two fermions (spin-up and spin-down) occupy the same site, they interact with energy $U$. The number operator $n_i = (I - Z_i)/2$ counts the occupation of qubit $i$, so the interaction at site $s$ is $n_{s,\uparrow} \cdot n_{s,\downarrow}$.

We use **6 qubits** for 3 sites × 2 spin states (spin-up and spin-down per site).

Below we construct the Hamiltonian and inspect its Pauli terms.

In [ ]:
# Build the Hamiltonian for J=1.0, U=0 (free-fermion case)
H = build_hamiltonian(J=1.0, U=0.0)

# Display the Pauli terms and coefficients
print("Fermi-Hubbard Hamiltonian (J=1.0, U=0)")
print("="*50)
print(f"Number of qubits: {H.num_qubits}")
print(f"Number of Pauli terms: {len(H.paulis)}")
print("\nPauli terms and coefficients:")
for pauli, coeff in zip(H.paulis, H.coeffs):
    print(f"  {str(pauli):8s}  {coeff.real:+.4f}")

# Verify the qubit count is 6
assert H.num_qubits == 6, f"Expected 6 qubits, got {H.num_qubits}"
print("\n✓ Verification passed: Hamiltonian has 6 qubits as expected.")

---
## Section 2: Exact Diagonalization ($U = 0$)

When $U = 0$, the Fermi-Hubbard model reduces to a free-fermion (tight-binding) model. The Hamiltonian contains only hopping terms and can be diagonalized exactly on a classical computer.

We convert the `SparsePauliOp` to a dense $64 \times 64$ matrix and use `numpy.linalg.eigvalsh` (which exploits Hermiticity for numerical stability) to find all eigenvalues. The minimum eigenvalue is the **exact ground state energy**.

For a 3-site chain with 2 fermions (one spin-up, one spin-down) and $J = 1.0$, the expected ground state energy is approximately $-2\sqrt{2} \approx -2.828$.

This exact value serves as our reference for validating VQE results.

In [ ]:
# Compute the exact ground state energy for J=1.0, U=0
exact_energy = exact_ground_energy(H)

print("Exact Ground State Energy (J=1.0, U=0)")
print("="*50)
print(f"Exact ground state energy: {exact_energy:.6f}")
print(f"Expected value: -2√2 ≈ {-2*np.sqrt(2):.6f}")
print(f"\nDifference from expected: {abs(exact_energy - (-2*np.sqrt(2))):.6e}")
print("\n✓ Exact diagonalization complete.")

---
## Section 3: VQE Setup & Single Run

### Ansatz Choice

We use **`EfficientSU2`** with `entanglement='linear'` and `reps=2` as our variational ansatz. This circuit applies layers of single-qubit $SU(2)$ rotations interleaved with CNOT gates connecting neighboring qubits.

**Why `EfficientSU2`?**
- It is hardware-efficient and has sufficient expressibility for the 3-site system.
- Linear entanglement mirrors the nearest-neighbor topology of the Fermi-Hubbard lattice.
- While it does not strictly conserve particle number, it is particle-number-*approximate* and works well in practice for this system size.
- A strictly particle-number-conserving ansatz (e.g., `UCCSD`) would require second-quantization mapping and additional dependencies, making it less suitable for a pedagogical notebook.

### Optimizer

We use **SLSQP** (Sequential Least Squares Programming), a gradient-based optimizer from `qiskit_algorithms`. It is well-suited for smooth, continuous optimization landscapes and typically converges faster than gradient-free methods (e.g., COBYLA) for statevector simulations where gradients can be computed accurately via the parameter-shift rule.

### Estimator

We use `StatevectorEstimator` from `qiskit.primitives`, which computes exact expectation values without shot noise — ideal for a pedagogical demonstration.

In [ ]:
# Create the ansatz
ansatz = make_ansatz(num_qubits=6, reps=2)

print("Ansatz Circuit")
print("="*50)
print(f"Number of qubits: {ansatz.num_qubits}")
print(f"Number of parameters: {ansatz.num_parameters}")
print(f"Entanglement: linear")
print(f"Repetitions: 2")
print("\n✓ Ansatz created successfully.")

# Create the optimizer
optimizer = SLSQP(maxiter=500)

# Run VQE
print("\nRunning VQE...")
vqe_result = run_vqe(hamiltonian=H, ansatz=ansatz, optimizer=optimizer, seed=42)

# Display results
print("\nVQE Results (J=1.0, U=0)")
print("="*50)
print(f"VQE ground state energy: {vqe_result.eigenvalue:.6f}")
print(f"Exact ground state energy: {exact_energy:.6f}")
print(f"Absolute error: {abs(vqe_result.eigenvalue - exact_energy):.6e}")
print(f"Relative error: {abs(vqe_result.eigenvalue - exact_energy) / abs(exact_energy) * 100:.4f}%")

# Check if within tolerance
rel_error = abs(vqe_result.eigenvalue - exact_energy) / abs(exact_energy)
if rel_error < 0.05:  # 5% tolerance
    print("\n✓ VQE converged successfully (within 5% of exact energy).")
else:
    print(f"\n⚠ VQE relative error ({rel_error*100:.2f}%) exceeds 5% tolerance.")

---
## Section 4: Convergence Study — Varying $J$

We now study how VQE performance varies as the hopping parameter $J$ changes. We sweep $J$ over 5 uniformly spaced values in $[1.0, 5.0]$ with $U = 0$.

For each $J$ value we:
1. Build the Hamiltonian $H(J, U=0)$
2. Compute the exact ground state energy via diagonalization
3. Run VQE and record the optimized energy
4. Compute the relative error $|E_{\text{VQE}} - E_{\text{exact}}| / |E_{\text{exact}}|$

Points where the relative error exceeds 1% are highlighted in the plot.

In [ ]:
# TODO: Run sweep_J and plot results (implemented in Section 5 task)


---
## Section 5: (Optional) Convergence Study — Varying $U$

In this optional section, we fix $J = 1.0$ and vary the on-site interaction $U \in \{0, 0.5, 1.0\}$.

As $U$ increases, the Hamiltonian gains interaction terms that introduce correlations between spin-up and spin-down fermions at the same site. We examine whether the `EfficientSU2` ansatz can still capture the ground state accurately in the presence of these interactions.

**Expected behavior:** For small $U$, VQE should converge well. For larger $U$, the ground state becomes more correlated and the ansatz may struggle, leading to larger relative errors.

In [ ]:
# TODO: Run sweep_U and plot results (implemented in Section 6 task)


---
## Section 6: Summary & Conclusions

### Summary of Results

*(To be filled in after running the notebook)*

- **Hamiltonian:** Successfully constructed the 6-qubit Fermi-Hubbard Hamiltonian as a `SparsePauliOp` with hopping and interaction terms.
- **Exact diagonalization:** Computed the exact ground state energy for $U=0$, confirming the expected free-fermion result.
- **VQE single run:** VQE with `EfficientSU2` (linear, reps=2) and SLSQP converged to within tolerance of the exact energy.
- **J-sweep:** VQE tracked the exact ground state energy across $J \in [1.0, 5.0]$, with relative errors reported.
- **U-sweep (optional):** VQE performance as a function of on-site interaction $U$ was assessed.

### Conclusions

The VQE algorithm with a hardware-efficient `EfficientSU2` ansatz provides a good approximation to the ground state energy of the 3-site Fermi-Hubbard model. The `StatevectorEstimator` enables exact expectation value computation, making this an ideal pedagogical demonstration of hybrid quantum-classical optimization.

Key takeaways:
- The Jordan-Wigner encoding maps the fermionic Hamiltonian to a qubit operator efficiently.
- `EfficientSU2` with linear entanglement is a practical ansatz for small lattice systems.
- SLSQP converges reliably for statevector simulations where the energy landscape is smooth.
- Increasing $U$ introduces stronger correlations that may challenge the ansatz expressibility.

In [ ]:
# End of notebook
print("Fermi-Hubbard VQE notebook complete.")